<a href="https://colab.research.google.com/github/Shashini294/Statistical-Learning-e22294/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Automated Engineering Framework - Data Diagnostic & Pipeline Suite
Optimized for integrity, structural clarity, and advanced processing.
"""

import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder

class EngineeringDataEngine:
    """
    A specialized analytical module developed for dataset remediation,
    statistical auditing, and interactive telemetry visualization.
    """
    def __init__(self):
        self.raw_dataset = None
        self.continuous_vars = []
        self.categorical_vars = []

    # === 1. DATA ACQUISITION & INGESTION ===
    def import_target_file(self):
        """Orchestrates interactive local file upload sequences inside runtime environment."""
        print("Select target CSV file for processing:")
        received_files = files.upload()
        if not received_files:
            print("Operation aborted. No valid file uploaded.")
            return

        active_filename = list(received_files.keys())[0]
        # Array of anomalous literal strings designated for null value conversion
        anomalous_literals = ['?', '$n/a$', 'n/a', 'N/A', 'NULL', 'null', ' ']

        self.raw_dataset = pd.read_csv(io.BytesIO(received_files[active_filename]), na_values=anomalous_literals)
        print(f"\nSuccessfully initialized: {active_filename}!")
        self._execute_type_inference()

    def _execute_type_inference(self):
        """Scans columns and safely forces numerical casting where structurally viable."""
        if self.raw_dataset is None: return

        for column_heading in self.raw_dataset.columns:
            cast_attempt = pd.to_numeric(self.raw_dataset[column_heading], errors='coerce')
            if not cast_attempt.isna().all():
                self.raw_dataset[column_heading] = cast_attempt

        # Segregate metric categories
        self.continuous_vars = list(self.raw_dataset.select_dtypes(include=[np.number]).columns)
        self.categorical_vars = list(self.raw_dataset.select_dtypes(exclude=[np.number]).columns)

    # === 2. QUALITY CONTROL & HYGIENE ===
    def generate_structural_audit(self):
        """Outputs high-level dimensional metrics and a segment preview of the record matrix."""
        if self.raw_dataset is None:
            print("Active workspace is vacant.")
            return

        total_rows, total_cols = self.raw_dataset.shape
        print(f"Matrix Dimension: {total_rows} records across {total_cols} attributes")
        print(f"Continuous Metrics ({len(self.continuous_vars)}): {self.continuous_vars}")
        print(f"Categorical Attributes ({len(self.categorical_vars)}): {self.categorical_vars}\n")
        print("--- Structural Sample Display (Top 20 Records) ---")
        display(self.raw_dataset.head(20))

    def impute_missing_records(self, approach='median', default_constant=None):
        """Resolves unassigned cell values using statistical central tendencies or custom constants."""
        if self.raw_dataset is None: return

        for field in self.raw_dataset.columns:
            if self.raw_dataset[field].isna().sum() == 0:
                continue

            if field in self.continuous_vars:
                if approach == 'mean':
                    self.raw_dataset[field].fillna(self.raw_dataset[field].mean(), inplace=True)
                elif approach == 'median':
                    self.raw_dataset[field].fillna(self.raw_dataset[field].median(), inplace=True)
                elif approach == 'mode':
                    self.raw_dataset[field].fillna(self.raw_dataset[field].mode()[0], inplace=True)
                elif approach == 'constant' and default_constant is not None:
                    self.raw_dataset[field].fillna(default_constant, inplace=True)
            else:
                frequency_distribution = self.raw_dataset[field].mode()
                fallback = frequency_distribution[0] if not frequency_distribution.empty else "Unspecified"
                self.raw_dataset[field].fillna(fallback if approach != 'constant' else default_constant, inplace=True)
        print(f"Dataset blank fields populated utilizing the '{approach}' methodology.")

    def purge_redundant_records(self):
        """Identifies and drops identical rows within the current DataFrame configuration."""
        if self.raw_dataset is not None:
            baseline_count = self.raw_dataset.shape[0]
            self.raw_dataset.drop_duplicates(inplace=True)
            print(f"Purged {baseline_count - self.raw_dataset.shape[0]} duplicate rows from memory.")

    def isolate_and_filter_outliers(self, target_fields=None, execute_deletion=True):
        """Employs Interquartile Range boundaries to identify or eliminate statistical anomalies."""
        if self.raw_dataset is None: return
        evaluation_targets = target_fields if target_fields else self.continuous_vars

        flagged_indices = set()
        for attribute in evaluation_targets:
            first_quartile = self.raw_dataset[attribute].quantile(0.25)
            third_quartile = self.raw_dataset[attribute].quantile(0.75)
            spread = third_quartile - first_quartile
            lower_threshold = first_quartile - 1.5 * spread
            upper_threshold = third_quartile + 1.5 * spread

            outliers = self.raw_dataset[(self.raw_dataset[attribute] < lower_threshold) | (self.raw_dataset[attribute] > upper_threshold)].index
            if execute_deletion:
                flagged_indices.update(outliers)
            else:
                print(f"Attribute '{attribute}': {len(outliers)} anomalies tracked outside limits ({lower_threshold:.2f}, {upper_threshold:.2f})")

        if execute_deletion and flagged_indices:
            self.raw_dataset.drop(index=list(flagged_indices), inplace=True)
            print(f"Successfully ejected {len(flagged_indices)} rows containing out-of-bounds variations.")

    def drop_selected_columns(self):
        """Allows direct column elimination based on comma-separated user inputs."""
        user_input = input("Enter field names to eliminate (comma-separated): ")
        kill_list = [token.strip() for token in user_input.split(',') if token.strip() in self.raw_dataset.columns]
        self.raw_dataset.drop(columns=kill_list, inplace=True)
        self._execute_type_inference()
        print(f"Eliminated attributes: {kill_list}")

    def drop_selected_rows(self):
        """Allows specific row elimination based on comma-separated numerical indices."""
        user_input = input("Enter entry indices to eliminate (comma-separated): ")
        try:
            kill_list = [int(token.strip()) for token in user_input.split(',') if token.strip().isdigit()]
            self.raw_dataset.drop(index=kill_list, inplace=True)
            print(f"Eliminated indices: {kill_list}")
        except Exception as system_error:
            print(f"Execution failed regarding index termination: {system_error}")

    # === 3. MACHINE LEARNING DATA PREPARATION ===
    def transform_continuous_features(self, transformation_mode='standard'):
        """Applies normalization engines like MinMax, Z-score, or Robust scales to numerical data."""
        if self.raw_dataset is None or not self.continuous_vars: return pd.DataFrame()

        scaling_engine = {
            'minmax': MinMaxScaler(),
            'robust': RobustScaler()
        }.get(transformation_mode, StandardScaler())

        normalized_array = scaling_engine.fit_transform(self.raw_dataset[self.continuous_vars])
        return pd.DataFrame(normalized_array, columns=self.continuous_vars, index=self.raw_dataset.index)

    def transform_categorical_features(self, encoding_mode='onehot'):
        """Translates categorical descriptors using One-Hot encoding or ordinal scalar steps."""
        if self.raw_dataset is None or not self.categorical_vars: return pd.DataFrame()

        if encoding_mode == 'onehot':
            conversion_engine = OneHotEncoder(sparse_output=False, drop='first')
            encoded_array = conversion_engine.fit_transform(self.raw_dataset[self.categorical_vars].astype(str))
            derived_labels = conversion_engine.get_feature_names_out(self.categorical_vars)
            return pd.DataFrame(encoded_array, columns=derived_labels, index=self.raw_dataset.index)

        elif encoding_mode in ['ordinal', 'uniform']:
            conversion_engine = OrdinalEncoder()
            encoded_array = conversion_engine.fit_transform(self.raw_dataset[self.categorical_vars].astype(str))
            mapped_dataframe = pd.DataFrame(encoded_array, columns=self.categorical_vars, index=self.raw_dataset.index)
            if encoding_mode == 'uniform':
                mapped_dataframe = (mapped_dataframe - mapped_dataframe.min()) / (mapped_dataframe.max() - mapped_dataframe.min() + 1e-9)
            return mapped_dataframe

    def construct_integrated_pipeline_df(self, numeric_strategy='standard', category_strategy='onehot'):
        """Assembles continuous and categorical vectors into a single structural matrix for deployment."""
        numeric_component = self.transform_continuous_features(transformation_mode=numeric_strategy)
        category_component = self.transform_categorical_features(encoding_mode=category_strategy)
        return pd.concat([numeric_component, category_component], axis=1)

    # === 4. DATA TELEMETRY VISUALIZATION ===
    def generate_univariate_plots(self, target_variables):
        """Displays distribution traits through grouped Violin, Scatter, and Histogram views."""
        for parameter in target_variables:
            if parameter not in self.continuous_vars: continue

            dashboard = make_subplots(rows=1, cols=3, subplot_titles=('Violin Geometry', 'Index Mapping', 'Frequency Density'))
            dashboard.add_trace(go.Violin(x=self.raw_dataset[parameter], box_visible=True, points='all', name=parameter), row=1, col=1)
            dashboard.add_trace(go.Scatter(y=self.raw_dataset[parameter], mode='markers', marker=dict(opacity=0.6)), row=1, col=2)
            dashboard.add_trace(go.Histogram(x=self.raw_dataset[parameter]), row=1, col=3)

            dashboard.update_layout(title_text=f"Telemetry Profile: {parameter}", showlegend=False, height=420)
            dashboard.show()

    def generate_bivariate_analysis(self, factor_a, factor_b):
        """Evaluates metadata behaviors to automatically implement corresponding analytical graphs."""
        if factor_a not in self.raw_dataset.columns or factor_b not in self.raw_dataset.columns: return

        is_a_numeric = factor_a in self.continuous_vars
        is_b_numeric = factor_b in self.continuous_vars

        if is_a_numeric and is_b_numeric:
            render_fig = px.scatter(self.raw_dataset, x=factor_a, y=factor_b, trendline="ols", title=f"Trend Evaluation: {factor_a} vs {factor_b}")
        elif not is_a_numeric and not is_b_numeric:
            render_fig = px.density_heatmap(self.raw_dataset, x=factor_a, y=factor_b, text_auto=True, title=f"Matrix Intersect: {factor_a} vs {factor_b}")
        else:
            grouping_var = factor_a if not is_a_numeric else factor_b
            measured_var = factor_b if not is_a_numeric else factor_a
            render_fig = px.box(self.raw_dataset, x=grouping_var, y=measured_var, points="all", title=f"Structural Spread: {measured_var} split by {grouping_var}")
        render_fig.show()

    # === 5. COGNITIVE STATISTICAL MAPPING ===
    def render_association_matrix(self):
        """Computes structural cross-correlations across varied feature states."""
        if self.raw_dataset is None: return
        labels = self.raw_dataset.columns
        dimension = len(labels)
        correlation_matrix = pd.DataFrame(np.zeros((dimension, dimension)), index=labels, columns=labels)

        for row_idx in labels:
            for col_idx in labels:
                if row_idx == col_idx:
                    correlation_matrix.loc[row_idx, col_idx] = 1.0
                elif row_idx in self.continuous_vars and col_idx in self.continuous_vars:
                    correlation_matrix.loc[row_idx, col_idx] = self.raw_dataset[row_idx].corr(self.raw_dataset[col_idx], method='pearson')
                elif row_idx in self.categorical_vars and col_idx in self.categorical_vars:
                    cross_tabulation = pd.crosstab(self.raw_dataset[row_idx], self.raw_dataset[col_idx])
                    if cross_tabulation.size > 0:
                        from scipy.stats import chi2_contingency
                        chi2_value = chi2_contingency(cross_tabulation)[0]
                        sample_size = cross_tabulation.sum().sum()
                        phi_squared = chi2_value / sample_size
                        num_rows, num_cols = cross_tabulation.shape
                        phi_corr = max(0, phi_squared - ((num_cols-1)*(num_rows-1))/(sample_size-1))
                        row_corr = num_rows - ((num_rows-1)**2)/(sample_size-1)
                        col_corr = num_cols - ((num_cols-1)**2)/(sample_size-1)
                        correlation_matrix.loc[row_idx, col_idx] = np.sqrt(phi_corr / min((col_corr-1), (row_corr-1))) if min((col_corr-1), (row_corr-1)) > 0 else 0
                else:
                    continuous_target = row_idx if row_idx in self.continuous_vars else col_idx
                    categorical_target = col_idx if row_idx in self.continuous_vars else row_idx
                    try:
                        numeric_codes = self.raw_dataset[categorical_target].astype('category').cat.codes
                        correlation_matrix.loc[row_idx, col_idx] = self.raw_dataset[continuous_target].corr(numeric_codes)
                    except:
                        correlation_matrix.loc[row_idx, col_idx] = 0.0

        render_fig = px.imshow(correlation_matrix, text_auto=".2f", color_continuous_scale='Magma',
                        title="Unified Feature Association Telemetry Map")
        render_fig.show()

class RenderingPipeline:
    """Handles independent lower-level graphical generation sequences returning sandboxed code elements."""

    def __init__(self):
        pass

    def build_bar_chart(self, independent_var, dependent_var, source_data, color_factor=None, display_mode='group', header_text=None):
        """Assembles a modular bar presentation component converted into raw HTML text blocks."""
        try:
            render_fig = px.bar(source_data, x=independent_var, y=dependent_var, color=color_factor, barmode=display_mode, title=header_text, text_auto=True)
            return {"execution_state": "success", "html_payload": render_fig.to_html(include_plotlyjs='cdn', full_html=False)}
        except Exception as runtime_error:
            return {"execution_state": "error", "fault_log": str(runtime_error)}

    def build_pie_chart(self, categoric_labels, continuous_values, source_data, inner_radius=0.4, header_text=None):
        """Assembles a modular balanced donut component representation."""
        try:
            render_fig = px.pie(source_data, names=categoric_labels, values=continuous_values, hole=inner_radius, title=header_text)
            return {"execution_state": "success", "html_payload": render_fig.to_html(include_plotlyjs='cdn', full_html=False)}
        except Exception as runtime_error:
            return {"execution_state": "error", "fault_log": str(runtime_error)}

    def execute_notebook_render(self, execution_bundle):
        """Safely parses and renders standard HTML outputs generated via internal layout configurations."""
        if execution_bundle.get("execution_state") == "success":
            from IPython.display import HTML, display
            display(HTML(execution_bundle["html_payload"]))
        else:
            print(f"Rendering Exception Encounted: {execution_bundle.get('fault_log')}")

# ==========================================
# SYSTEM VALIDATION / PIPELINE EXECUTION FLOW
# ==========================================

# Initialize modular blocks
analytical_engine = EngineeringDataEngine()
visualization_terminal = RenderingPipeline()

# 1. Ingest structural verification records (Titanic Suite)
verification_source_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
analytical_engine.raw_dataset = pd.read_csv(verification_source_url)
analytical_engine._execute_type_inference()

# 2. Structural Inspection Evaluation
analytical_engine.generate_structural_audit()

# 3. Clean and isolate empty items
analytical_engine.impute_missing_records(approach='median')
analytical_engine.purge_redundant_records()

# 4. Outlier analysis execution pass
analytical_engine.isolate_and_filter_outliers(target_fields=['Age', 'Fare'], execute_deletion=False)

# 5. Extract finalized training matrix layout
processed_feature_matrix = analytical_engine.construct_integrated_pipeline_df(numeric_strategy='robust', category_strategy='onehot')
print("\n--- Pipeline Transformation Matrix Sample Output ---")
display(processed_feature_matrix.head(5))

# 6. Comprehensive Graph Generations
analytical_engine.generate_univariate_plots(['Age', 'Fare'])
analytical_engine.generate_bivariate_analysis('Pclass', 'Fare')
analytical_engine.render_association_matrix()

# 7. Execute sandboxed secondary component renderings
graph_package = visualization_terminal.build_pie_chart(categoric_labels='Sex', continuous_values='PassengerId', source_data=analytical_engine.raw_dataset, header_text='Demographic Metric Breakdown')
visualization_terminal.execute_notebook_render(graph_package)